In [1]:
!pip install -q datasets transformers

In [2]:
import torch
import re
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# CONFIG
# =========================

MODEL_NAME = "microsoft/phi-3-mini-4k-instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_NEW_TOKENS = 128
NUM_SAMPLES = 50
NUM_GENERATIONS = 3

# =========================
# LOAD MODEL
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto"
)

model.eval()

# =========================
# PROMPT BUILDER
# =========================

def build_prompt(question, mode="cot"):
    if mode == "plain":
        content = question
    elif mode == "cot":
        content = f"Solve step-by-step:\n\n{question}"
    elif mode == "structured":
        content = f"""Solve carefully:

<think>
Step-by-step reasoning
</think>
<answer>
Final answer
</answer>

Question: {question}
"""
    else:
        content = question

    messages = [
        {"role": "system", "content": "You are a reasoning assistant."},
        {"role": "user", "content": content}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

# =========================
# UTILS
# =========================

def extract_number(text):
    matches = re.findall(r"[-+]?\d*\.\d+|\d+", text)
    return matches[-1] if matches else None

def extract_yes_no(text):
    text = text.lower()
    if "yes" in text:
        return "yes"
    elif "no" in text:
        return "no"
    return None

def extract_mcq(text):
    text = text.upper()
    for choice in ["A", "B", "C", "D"]:
        if f"{choice}" in text:
            return choice
    return None

def format_compliance(text):
    return int("<think>" in text and "<answer>" in text)

# =========================
# GENERATION
# =========================

def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

def multi_generate(prompt):
    outputs, answers = [], []

    for _ in range(NUM_GENERATIONS):
        out = generate(prompt)
        outputs.append(out)

    return outputs

def majority_vote(preds):
    preds = [p for p in preds if p is not None]
    if not preds:
        return None
    return max(set(preds), key=preds.count)

# =========================
# STRATEGYQA BENCHMARK
# =========================

def run_strategyqa(dataset, mode):
    results = []

    for sample in tqdm(dataset):
        question = sample["question"]
        gt = "yes" if sample["answer"] else "no"

        prompt = build_prompt(question, mode)
        outputs = multi_generate(prompt)

        preds = [extract_yes_no(o) for o in outputs]
        final_pred = majority_vote(preds)

        correct = final_pred == gt if final_pred else False

        results.append({
            "question": question,
            "ground_truth": gt,
            "prediction": final_pred,
            "correct": correct,
            "avg_length": sum(len(o.split()) for o in outputs)/len(outputs),
            "format_rate": sum(format_compliance(o) for o in outputs)/len(outputs)
        })

    df = pd.DataFrame(results)

    print(f"\n=== StrategyQA ({mode}) ===")
    print("Accuracy:", df["correct"].mean())
    print("Avg Length:", df["avg_length"].mean())
    print("Format Compliance:", df["format_rate"].mean())

    return df

# =========================
# MMLU BENCHMARK
# =========================

def run_mmlu(dataset, mode):
    results = []

    for sample in tqdm(dataset):
        question = sample["question"]
        choices = sample["choices"]
        gt = chr(65 + sample["answer"])  # convert index → A/B/C/D

        formatted_q = question + "\n"
        for i, c in enumerate(choices):
            formatted_q += f"{chr(65+i)}. {c}\n"

        prompt = build_prompt(formatted_q, mode)
        outputs = multi_generate(prompt)

        preds = [extract_mcq(o) for o in outputs]
        final_pred = majority_vote(preds)

        correct = final_pred == gt if final_pred else False

        results.append({
            "question": question,
            "ground_truth": gt,
            "prediction": final_pred,
            "correct": correct,
            "avg_length": sum(len(o.split()) for o in outputs)/len(outputs),
            "format_rate": sum(format_compliance(o) for o in outputs)/len(outputs)
        })

    df = pd.DataFrame(results)

    print(f"\n=== MMLU ({mode}) ===")
    print("Accuracy:", df["correct"].mean())
    print("Avg Length:", df["avg_length"].mean())
    print("Format Compliance:", df["format_rate"].mean())

    return df

# =========================
# LOAD DATASETS
# =========================

strategyqa = load_dataset("ChilleD/StrategyQA", split="test").select(range(NUM_SAMPLES))
mmlu = load_dataset("cais/mmlu", "abstract_algebra", split="test").select(range(NUM_SAMPLES))

# =========================
# RUN
# =========================

strat_df = run_strategyqa(strategyqa, mode="cot")
mmlu_df = run_mmlu(mmlu, mode="cot")

strat_df.to_csv("strategyqa_results.csv", index=False)
mmlu_df.to_csv("mmlu_results.csv", index=False)

print("\nSaved all results.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/433 [00:00<?, ?B/s]

data/train-00000-of-00001-506370352f6228(…):   0%|          | 0.00/369k [00:00<?, ?B/s]

data/test-00000-of-00001-bae602f3ee37f4c(…):   0%|          | 0.00/161k [00:00<?, ?B/s]

...

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

100%|██████████| 50/50 [12:43<00:00, 15.27s/it]



=== StrategyQA (cot) ===
Accuracy: 0.32
Avg Length: 98.42
Format Compliance: 0.0


100%|██████████| 50/50 [12:44<00:00, 15.29s/it]


=== MMLU (cot) ===
Accuracy: 0.2
Avg Length: 125.3
Format Compliance: 0.0

Saved all results.
